# CRIS-SME Methodology and Results Notebook

This notebook turns the current live report and archived run history into a compact methodology and results workspace suitable for research drafting, figure preparation, and repeatable demo analysis.

In [ ]:
import sys
from pathlib import Path
import json

repo_root = Path('..').resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cris_sme.reporting import load_report_history

report_path = repo_root / 'outputs' / 'reports' / 'cris_sme_report.json'
history_dir = repo_root / 'outputs' / 'reports' / 'history'
figures_dir = repo_root / 'outputs' / 'figures'

current_report = json.loads(report_path.read_text(encoding='utf-8'))
history_reports = load_report_history(history_dir)

len(history_reports), current_report['overall_risk_score']

## Methodology Summary

- Collector mode is recorded per run in the archived report snapshots.
- The risk model is deterministic and explainable, with category scores plus prioritized findings.
- Identity evidence is explicitly annotated with observability limits so that the results do not overclaim tenant-wide Entra coverage.

In [ ]:
print('Current report generated at:', current_report.get('generated_at'))
print('Collector mode:', current_report.get('collector_mode'))
print('Overall risk score:', current_report.get('overall_risk_score'))
print('Evaluation context:', current_report.get('evaluation_context'))
print('Category scores:', current_report.get('category_scores'))

In [ ]:
top_risks = current_report.get('prioritized_risks', [])[:8]
[(item['control_id'], item['category'], item['score'], item['priority']) for item in top_risks]

## Run Deltas

The report now includes both immediate previous-run deltas and deltas versus the most recent different collector mode, which is useful for comparing mock and live runs.

In [ ]:
current_report.get('history_comparison')

In [ ]:
current_report.get('history_comparison', {}).get('control_score_deltas_vs_distinct_mode', [])[:10]

## Archived Run Overview

The history directory can be used to compare repeated CRIS-SME executions over time. This is especially useful once the project begins collecting assessments from multiple dates, subscriptions, or provider implementations.

In [ ]:
[
    (
        report.get('generated_at'),
        report.get('collector_mode'),
        report.get('overall_risk_score')
    )
    for report in history_reports[-10:]
]

## Figure References

These figure assets are generated into `outputs/figures/` and can be embedded directly into markdown docs, slides, and conference material.

In [ ]:
sorted(str(path.relative_to(repo_root)) for path in figures_dir.glob('*.svg'))

## Appendix Tables

CRIS-SME now exports appendix-friendly tables alongside the core report so results can be reused directly in papers, appendices, and spreadsheet workflows.

In [ ]:
appendix_dir = repo_root / 'outputs' / 'reports'
sorted(str(path.relative_to(repo_root)) for path in appendix_dir.glob('*.md')) + sorted(str(path.relative_to(repo_root)) for path in appendix_dir.glob('*.csv'))

In [ ]:
appendix_path = appendix_dir / 'results_appendix.md'
if appendix_path.exists():
    print(appendix_path.read_text(encoding='utf-8')[:2500])

## Suggested Research Write-up Structure

1. Describe the Azure-first but provider-neutral CRIS-SME architecture.
2. Explain that the scoring engine is deterministic and intentionally interpretable.
3. Present the current live Azure case study with archived run metadata and figures.
4. Discuss identity observability as an explicit methodological boundary rather than a hidden limitation.
5. Use archived runs to compare score movement as the collector matures.